In [ ]:
# %%
from pathlib import Path
import os, sys
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)
print("Project root:", project_root)
from eintelligence.data_prep.aoi import square_aoi
from orchestrator.workflow_manager_multisensor_v1 import (
    DeforestationWorkflowMS, TilingConfigMS, TilingConfigS1, TrainingConfig,
    FloodWorkflowS1
)


CASE = 1

# 0 - DEFORESTATION - S2 
# 1 - DEFORESTSTION - MULTISENSOR
# 2 - FLOOD - S1



Project root: /home/hegde/code/earth-api


In [ ]:
aoi = square_aoi(48.1351, 11.5820)
# aoi = square_aoi(-7.754, -55.513)  # Novo Progresso (Pará, BR-163 / Jamanxim front)

# Pick ONE:
# sensor_mode = "s2"
sensor_mode = "s1"
# sensor_mode = "s1s2"

if CASE == 1:
    case_name = "deforestation"
    
    tiling_cfg = TilingConfigMS(
        bands_s2=("B02","B03","B04","B08"),
        bands_s1=("vv","vh"),
        tile_size=256, stride=256, max_cloud=50,
        sensor_mode=sensor_mode
    )

    train_cfg  = TrainingConfig(batch_size=4, num_epochs=10, lr=1e-3, amp=True)

    wf = DeforestationWorkflowMS(project_root, tiling_cfg, train_cfg, skip_to_pairing=True)

elif CASE == 2:
    case_name = "flood"
    tiling_cfg = TilingConfigS1(tile_size=256, stride=256)

    train_cfg  = TrainingConfig(batch_size=4, num_epochs=10, lr=1e-3, amp=True)
    
    wf = FloodWorkflowS1(project_root, tiling_cfg, train_cfg, skip_to_pairing=False)

print("The case is - ", case_name)


/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:550: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = GradScaler(enabled=(device.type=="cuda" and cfg.amp))


The case is -  flood


In [ ]:


region_name = f"munich_{case_name}_{sensor_mode}"
pairs_manifest = wf.build_data(
    aoi_geojson=aoi,
    start="2023-06-01",
    end="2023-08-01",
    region_name=region_name
)

ckpt_path = Path(project_root) / "models" / f"{case_name}_{sensor_mode}_adapter.pt"
out_dir   = Path(project_root) / "data" / region_name / f"pred_{case_name}_{sensor_mode}"

wf.run(pairs_manifest, ckpt_path, out_dir, retrain=False, prob_thresh=0.5)

S1 pairs manifest: /home/hegde/code/earth-api/data/brazil_flood_s1/S1/pairs_manifest.json


/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:592: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type=="cuda" and self.cfg.amp)):


[epoch 00] train=0.7129  val=0.1202
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 01] train=0.0610  val=0.0805
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 02] train=0.0363  val=0.0582
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 03] train=0.0268  val=0.0496
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 04] train=0.0240  val=0.0485
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 05] train=0.0235  val=0.0478
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 06] train=0.0232  val=0.0474
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 07] train=0.0231  val=0.0471
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 08] train=0.0230  val=0.0470
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt
[epoch 09] train=0.0229  val=0.0467
  ↳ saved /home/hegde/code/earth-api/models/flood_s1_adapter.pt


/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:712: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type=="cuda")):
/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:712: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type=="cuda")):
/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:712: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self.device.type=="cuda")):
/home/hegde/code/earth-api/orchestrator/workflow_manager_multisensor_v1.py:712: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(self

wrote 16 flood tiles -> /home/hegde/code/earth-api/data/brazil_flood_s1/pred_flood_s1


PosixPath('/home/hegde/code/earth-api/data/brazil_flood_s1/pred_flood_s1')